# 미국 수출입 YoY "가속도" vs 전체 티커 주가 재스크리닝

이전 v1 스크리닝(YoY 성장률 "수준")은 정밀검증 결과 대부분 추세 공유로 인한 착시로 판명됐습니다.
이번에는 처음부터 **YoY 가속도(전월 대비 YoY 변화폭, 이미 추세 제거됨)** 를 기준으로,
60개 HS 코드(수출 상위 30 + 수입 상위 30, 500% 이상 폭주 예측 제외) × 전체 티커 × 3개 horizon을
**HAC(Newey-West) 보정 유의성 검정**으로 한 번에 스크리닝합니다.

## v1과의 차이
| | v1 (레벨 스크리닝) | 이번 (가속도 스크리닝) |
|---|---|---|
| x변수 | YoY 성장률 수준 | YoY 성장률 가속도 (`.diff(1)`) |
| 유의성 판정 | 단순 Pearson r만 계산 | 애초에 HAC 보정 t검정으로 스크리닝 |
| 다중비교 보정 | 사후에 별도 확인 필요 | 스크리닝 결과 전체에 BH-FDR 바로 적용 |
| 발표시차 반영 | 동일 (2.5개월 반영) | 동일 |

## 주의
- 이 스크리닝을 통과(FDR 유의)했다고 바로 "실전 시그널"은 아닙니다. 지난 LLY 사례처럼
  **섹터 공통효과 여부 / out-of-sample 검증**은 아직 거치지 않았습니다.
  여기서 나온 후보는 반드시 개별적으로 2차 정밀검증(OOS, 섹터컨트롤)을 거쳐야 합니다.


In [1]:

# -*- coding: utf-8 -*-
from __future__ import annotations
import os, sys
from pathlib import Path


def _find_project_root(module_name="DATA", max_up=6):
    here = Path.cwd()
    for base in [here, *list(here.parents)[:max_up]]:
        if (base / module_name).is_dir():
            return base
    return None


try:
    from DATA.stock_invest_function import *
except ModuleNotFoundError:
    _root = _find_project_root("DATA")
    if _root is None:
        raise ModuleNotFoundError("DATA 패키지를 찾을 수 없습니다. 프로젝트 루트에서 실행해 주세요.")
    sys.path.insert(0, str(_root))
    from DATA.stock_invest_function import *
    print(f"[경로 자동 보정] DATA 모듈을 {_root} 에서 찾아 sys.path에 추가했습니다.")

import numpy as np
import pandas as pd
from scipy import stats
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore")

db_info = {"host": get_db_host(), "port": 3307, "user": "stox7412",
           "password": "Apt106503!~", "database": "investar"}


def make_engine(db_info):
    url = (f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
           f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4")
    return create_engine(url, pool_pre_ping=True, pool_recycle=1800,
                          connect_args={"connect_timeout": 10, "read_timeout": 60, "write_timeout": 60})


engine = make_engine(db_info)
TABLE_EXPORT = "us_trade_export_monthly_with_forecast"
TABLE_IMPORT = "us_trade_import_monthly_with_forecast"
TABLE_PRICE  = "us_stock_daily_market_cap"


In [2]:

# ============================================================
# 데이터 로딩 헬퍼
# ============================================================

def _month_end(ts):
    return pd.Timestamp(ts) + pd.offsets.MonthEnd(0)


def _detect_version_col(conn, table):
    res = conn.execute(text(f"SHOW COLUMNS FROM {table}"))
    cols = [r[0] for r in res.fetchall()]
    for c in ("created_at", "input_date"):
        if c in cols:
            return c
    return None


def compute_publication_date(ref_month_end, lag_months=2, lag_day=15):
    ref_month_end = pd.Timestamp(ref_month_end)
    return (ref_month_end + pd.DateOffset(months=lag_months)).replace(day=lag_day)


def _load_latest_export_data(engine, table=TABLE_EXPORT):
    with engine.connect() as conn:
        vcol = _detect_version_col(conn, table)
        if vcol:
            sql = f"""
                SELECT t.hs_code AS hs_code, t.date_month_end AS date, t.expDlr, t.expDlr_forecast, t.is_forecast
                FROM {table} t
                JOIN (SELECT hs_code, MAX({vcol}) AS mx FROM {table} GROUP BY hs_code) l
                  ON t.hs_code = l.hs_code AND t.{vcol} = l.mx
                ORDER BY t.hs_code, t.date_month_end
            """
        else:
            sql = f"SELECT hs_code, date_month_end AS date, expDlr, expDlr_forecast, is_forecast FROM {table}"
        rows = conn.execute(text(sql)).fetchall()
    df = pd.DataFrame(rows, columns=["hs_code", "date", "expDlr", "expDlr_forecast", "is_forecast"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = np.where(df["is_forecast"] == 1, df["expDlr_forecast"], df["expDlr"])
    df["forecast_flag"] = df["is_forecast"]
    df["hs_code"] = df["hs_code"].astype(str).str.zfill(6)
    return df[["hs_code", "date", "value", "forecast_flag"]]


def _load_latest_import_data(engine, table=TABLE_IMPORT):
    with engine.connect() as conn:
        latest_vintage = conn.execute(text(f"SELECT MAX(input_date) FROM {table}")).scalar()
        rows = conn.execute(text(f"""
            SELECT hs_code_6d AS hs_code, date, impDlr AS value, forecast_flag
            FROM {table} WHERE input_date = :v ORDER BY hs_code_6d, date
        """), {"v": latest_vintage}).fetchall()
    df = pd.DataFrame(rows, columns=["hs_code", "date", "value", "forecast_flag"])
    df["date"] = pd.to_datetime(df["date"])
    df["hs_code"] = df["hs_code"].astype(str).str.zfill(6)
    return df


def analyze_growth_ranking(df_all, base_date, sanity_ratio=1000.0):
    base_dt = pd.to_datetime(base_date)
    past_start   = _month_end(base_dt - pd.DateOffset(months=11))
    future_start = _month_end(base_dt + pd.DateOffset(months=1))
    future_end   = _month_end(future_start + pd.DateOffset(months=11))
    rows = []
    for hs, df_hs in df_all.groupby("hs_code"):
        past = df_hs.loc[(df_hs["date"] >= past_start) & (df_hs["date"] <= base_dt), "value"].sum()
        future = df_hs.loc[(df_hs["date"] >= future_start) & (df_hs["date"] <= future_end), "value"].sum()
        if pd.isna(past) or past <= 0 or pd.isna(future) or future < 0:
            continue
        rows.append(dict(hs_code=hs, past_12m_sum=past, future_12m_sum=future,
                          growth_rate=(future / past - 1.0) * 100.0))
    return pd.DataFrame(rows).sort_values("growth_rate", ascending=False).reset_index(drop=True)


def get_full_actual_yoy_series(df_all, hs_code):
    df_hs = df_all[(df_all["hs_code"] == hs_code) & (df_all["forecast_flag"] == 0)].copy()
    if df_hs.empty:
        return pd.Series(dtype=float)
    df_hs["date"] = df_hs["date"].apply(_month_end)
    ts = df_hs.groupby("date")["value"].sum().sort_index()
    return (ts.pct_change(12) * 100.0).dropna()


def get_yoy_acceleration_series(df_all, hs_code):
    """YoY 성장률의 가속도 = 전월 대비 YoY 변화폭 (이미 추세 제거됨)."""
    return get_full_actual_yoy_series(df_all, hs_code).diff(1).dropna()


def load_monthly_close_panel(engine, start_date, table=TABLE_PRICE):
    sql = text(f"""
        SELECT date, ticker, value FROM {table}
        WHERE indicator = 'close_price' AND date >= :start_date
    """)
    with engine.connect() as conn:
        rows = conn.execute(sql, {"start_date": start_date}).fetchall()
    df = pd.DataFrame(rows, columns=["date", "ticker", "value"])
    if df.empty:
        raise ValueError("가격 데이터가 없습니다.")
    df["date"] = pd.to_datetime(df["date"])
    daily = df.pivot_table(index="date", columns="ticker", values="value", aggfunc="last").sort_index()
    return daily.resample("ME").last()


def build_forward_return_panels(monthly_close, horizons=(3, 6, 12)):
    return {h: monthly_close.shift(-h) / monthly_close - 1.0 for h in horizons}


def align_accel_and_get_base_dates(x_series, monthly_close_index, lag_months=2, lag_day=15):
    """가속도 시계열을 발표시차 반영해 base_date(월말)로 스냅한 Series 반환."""
    pub_dates = pd.DatetimeIndex([compute_publication_date(d, lag_months, lag_day) for d in x_series.index])
    base_dates = pub_dates.map(lambda d: monthly_close_index.asof(d))
    valid = pd.notna(base_dates)
    g = pd.Series(x_series.values[valid], index=pd.DatetimeIndex(np.array(base_dates)[valid]))
    return g[~g.index.duplicated(keep="last")].sort_index()


In [3]:

# ============================================================
# 벡터화 HAC 검정 + 다중비교 보정
# ============================================================

def hac_test_matrix(x, Y, max_lag=None):
    """
    x: (n,) ndarray, Y: (n,k) ndarray (여러 티커 동시 처리)
    반환: r, beta, t_hac, p_hac (모두 (k,) 배열), n, max_lag
    """
    n = len(x)
    if max_lag is None:
        max_lag = max(int(np.floor(4 * (n / 100) ** (2 / 9))), 1)

    xbar = x.mean()
    xc = x - xbar
    xvar = (xc ** 2).sum()
    Yc = Y - Y.mean(axis=0, keepdims=True)
    b = (xc[:, None] * Yc).sum(axis=0) / xvar
    a = Y.mean(axis=0) - b * xbar
    resid = Y - (a[None, :] + b[None, :] * x[:, None])

    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)

    u = X[:, None, :] * resid[:, :, None]  # (n,k,2)
    S = np.einsum('tka,tkb->kab', u, u)
    for lag in range(1, max_lag + 1):
        w = 1 - lag / (max_lag + 1)
        Gamma = np.einsum('tka,tkb->kab', u[lag:], u[:-lag])
        S += w * (Gamma + np.transpose(Gamma, (0, 2, 1)))

    V = np.einsum('ab,kbc,cd->kad', XtX_inv, S, XtX_inv)
    se_b = np.sqrt(np.clip(V[:, 1, 1], 1e-12, None))
    t_hac = b / se_b
    p_hac = 2 * (1 - stats.t.cdf(np.abs(t_hac), max(n - 2, 1)))
    r = xc @ Yc / np.sqrt(xvar * (Yc ** 2).sum(axis=0))

    return dict(r=r, beta=b, t_hac=t_hac, p_hac=p_hac, n=n, max_lag=max_lag)


def benjamini_hochberg(pvalues, alpha=0.05):
    p = np.asarray(pvalues)
    n = len(p)
    order = np.argsort(p)
    ranked_p = p[order]
    adj = ranked_p * n / (np.arange(1, n + 1))
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    adj_full = np.empty(n)
    adj_full[order] = adj
    return adj_full <= alpha, adj_full


## ① 사용자 입력

In [4]:

TOP_N_HS             = 30
EXPLOSIVE_THRESHOLD  = 500.0
HORIZONS             = (3, 6, 12)
LAG_MONTHS, LAG_DAY  = 2, 15
PRICE_HISTORY_YEARS  = 6
MIN_OBS_FOR_TEST     = 20      # 가속도는 레벨보다 표본이 1개 더 적으므로 최소치도 확인
FDR_ALPHA            = 0.05
TOP_N_RESULT         = 50      # 최종 후보 리스트에서 보여줄 상위 개수
SAVE_DIR             = str(Path.home() / "reports" / "us_trade_stock_corr_accel")


## ② 대상 HS 코드 60개 선정 + 주가 패널 로드

In [5]:

export_all = _load_latest_export_data(engine, TABLE_EXPORT)
import_all = _load_latest_import_data(engine, TABLE_IMPORT)

export_base_date = export_all.loc[export_all["forecast_flag"] == 0, "date"].max()
import_base_date = import_all.loc[import_all["forecast_flag"] == 0, "date"].max()

export_ranking = analyze_growth_ranking(export_all, export_base_date)
import_ranking = analyze_growth_ranking(import_all, import_base_date)

export_target = export_ranking[export_ranking["growth_rate"] < EXPLOSIVE_THRESHOLD].head(TOP_N_HS).copy()
export_target["direction"] = "export"
import_target = import_ranking[import_ranking["growth_rate"] < EXPLOSIVE_THRESHOLD].head(TOP_N_HS).copy()
import_target["direction"] = "import"

target_hs_df = pd.concat([export_target, import_target], ignore_index=True)
print(f"분석 대상 HS 코드: {len(target_hs_df)}개 (수출 {len(export_target)} + 수입 {len(import_target)})")

price_start_date = (pd.Timestamp.today() - pd.DateOffset(years=PRICE_HISTORY_YEARS)).strftime("%Y-%m-%d")
monthly_close = load_monthly_close_panel(engine, price_start_date, TABLE_PRICE)
forward_panels = build_forward_return_panels(monthly_close, horizons=HORIZONS)

print(f"주가 데이터 기간: {price_start_date} ~ {monthly_close.index.max().date()}")
print(f"티커 수: {monthly_close.shape[1]:,}개")


분석 대상 HS 코드: 60개 (수출 30 + 수입 30)
주가 데이터 기간: 2020-07-24 ~ 2026-07-31
티커 수: 1,996개


## ③ 가속도 기준 전체 스크리닝 (HAC 보정, 벡터화)

In [6]:

all_results = []

for _, row in target_hs_df.iterrows():
    hs_code, direction = row["hs_code"], row["direction"]
    df_all = export_all if direction == "export" else import_all

    accel_series = get_yoy_acceleration_series(df_all, hs_code)
    if len(accel_series) < MIN_OBS_FOR_TEST:
        continue

    g = align_accel_and_get_base_dates(accel_series, monthly_close.index, LAG_MONTHS, LAG_DAY)
    if len(g) < MIN_OBS_FOR_TEST:
        continue

    for h in HORIZONS:
        ret_df = forward_panels[h].reindex(g.index)
        # 유효 관측치(x,y 둘 다 존재)가 부족한 티커는 결과 자체가 nan으로 나오므로 이후 dropna 처리
        Y = ret_df.values
        valid_cols = ~np.all(np.isnan(Y), axis=0)
        if valid_cols.sum() == 0:
            continue

        # 결측이 섞여 있으면 컬럼별 관측치 수가 달라 벡터화 계산이 어긋나므로,
        # 결측 없는(완전한) 티커만 우선 벡터화 처리하고 나머지는 개별 처리
        complete_mask = ~np.isnan(Y).any(axis=0)
        tickers_all = ret_df.columns.values

        if complete_mask.sum() > 0:
            xv = g.values.astype(float)
            Yc = Y[:, complete_mask].astype(float)
            mat = hac_test_matrix(xv, Yc)
            out = pd.DataFrame({
                "ticker": tickers_all[complete_mask],
                "r": mat["r"], "p_hac": mat["p_hac"], "n": mat["n"],
            })
            out["hs_code"], out["direction"], out["horizon"] = hs_code, direction, h
            out["future_12m_growth_rate"] = row["growth_rate"]
            all_results.append(out)

        # 결측이 일부 있는 티커는 개별적으로 dropna 후 처리 (수가 적으면 느려도 무방)
        partial_idx = np.where((~complete_mask) & valid_cols)[0]
        for j in partial_idx:
            yj = ret_df.iloc[:, j]
            pair = pd.DataFrame({"x": g, "y": yj}).dropna()
            if len(pair) < MIN_OBS_FOR_TEST:
                continue
            mat1 = hac_test_matrix(pair["x"].values, pair["y"].values.reshape(-1, 1))
            all_results.append(pd.DataFrame({
                "ticker": [tickers_all[j]], "r": mat1["r"], "p_hac": mat1["p_hac"], "n": mat1["n"],
                "hs_code": hs_code, "direction": direction, "horizon": h,
                "future_12m_growth_rate": row["growth_rate"],
            }))

screen_df = pd.concat(all_results, ignore_index=True)
screen_df = screen_df.dropna(subset=["r", "p_hac"])
print(f"전체 스크리닝 레코드: {len(screen_df):,}건 "
      f"(HS {screen_df['hs_code'].nunique()}개 x 티커 {screen_df['ticker'].nunique()}개 x horizon {len(HORIZONS)}개)")


전체 스크리닝 레코드: 347,366건 (HS 51개 x 티커 1949개 x horizon 3개)


## ④ 다중비교 보정 (Benjamini-Hochberg FDR, 전체 결과 기준)

In [7]:

rejected, adj_p = benjamini_hochberg(screen_df["p_hac"].values, alpha=FDR_ALPHA)
screen_df["p_adj_fdr"] = adj_p
screen_df["significant_after_fdr"] = rejected

n_sig = rejected.sum()
print(f"FDR {FDR_ALPHA*100:.0f}% 기준 통과: {n_sig:,}건 / 전체 {len(screen_df):,}건 ({n_sig/len(screen_df)*100:.3f}%)")

if n_sig == 0:
    print("\n[결과] FDR 보정을 통과한 조합이 하나도 없습니다.")
    print("       즉, 가속도 기준으로도 우연을 넘어서는 뚜렷한 시그널은 발견되지 않았습니다.")


FDR 5% 기준 통과: 830건 / 전체 347,366건 (0.239%)


## ⑤ 최종 후보 리스트

In [8]:

candidates = screen_df[screen_df["significant_after_fdr"]].copy()
candidates["abs_r"] = candidates["r"].abs()
candidates = candidates.sort_values("abs_r", ascending=False).head(TOP_N_RESULT)

cols = ["hs_code", "direction", "horizon", "ticker", "r", "n", "p_hac", "p_adj_fdr", "future_12m_growth_rate"]
with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
    print(candidates[cols].to_string(index=False))

os.makedirs(SAVE_DIR, exist_ok=True)
today_str = pd.Timestamp.today().strftime("%Y%m%d")
fp_full = os.path.join(SAVE_DIR, f"accel_screen_full_{today_str}.csv")
fp_cand = os.path.join(SAVE_DIR, f"accel_screen_candidates_fdr{int(FDR_ALPHA*100)}_{today_str}.csv")

screen_df.drop(columns="abs_r", errors="ignore").to_csv(fp_full, index=False, encoding="utf-8-sig")
candidates[cols].to_csv(fp_cand, index=False, encoding="utf-8-sig")

print(f"\n[저장] 전체 결과: {fp_full} ({len(screen_df):,} rows)")
print(f"[저장] FDR 통과 후보: {fp_cand} ({len(candidates):,} rows)")
print("\n[다음 단계] 여기서 나온 후보들은 반드시 개별적으로 out-of-sample 검증 + 섹터컨트롤 비교를 거친 뒤")
print("            실전 시그널 여부를 최종 판단하세요 (LLY_상관관계_정밀검증_v2 노트북의 ③~⑤단계 재사용 가능).")


hs_code direction  horizon ticker       r  n  p_hac  p_adj_fdr  future_12m_growth_rate
 870451    export        3   CNSL  0.7868 20 0.0000     0.0006                 49.2527
 870451    export       12   EYPT  0.7244 29 0.0000     0.0099                 49.2527
 850790    export        6   SBET -0.6981 67 0.0000     0.0036                 33.1393
 850790    export       12   SAND  0.6834 52 0.0000     0.0000                 33.1393
 870451    export        3   AZPN -0.6660 22 0.0000     0.0015                 49.2527
 870451    export       12    TAP  0.6633 29 0.0000     0.0000                 49.2527
 870451    export        3   DMAC  0.6222 37 0.0000     0.0000                 49.2527
 850790    import       12    CEA -0.5736 20 0.0000     0.0022                 29.7084
 870451    export        3   CNMD  0.5430 38 0.0000     0.0000                 49.2527
 847150    import        3    NRZ -0.5367 23 0.0001     0.0386                  4.6952
 870451    export       12   BBBY  0.5301 2